# 05 Model Comparison: IEEE-CIS Fraud Detection

Это `Jupyter notebook` для следующего шага после `03_baseline_model.ipynb` и `04_behavioral_features.ipynb`.

Цель:
- сравнить текущий честный baseline на `LogisticRegression` с еще одной простой моделью;
- проверить, есть ли в данных полезный нелинейный сигнал;
- не усложнять решение раньше времени и сохранить MVP-подход;
- интерпретировать результат не только по `roc_auc`, но и по anti-fraud метрикам.


## План работы

1. Загрузить `train_transaction` и `train_identity`.
2. Собрать тот же честный baseline feature set.
3. Разделить данные на `train / validation`.
4. Обучить `LogisticRegression` как reference model.
5. Обучить еще одну простую модель для сравнения.
6. Сравнить `precision`, `recall`, `f1`, `roc_auc` и `manual_review_rate`.
7. Коротко зафиксировать, какая модель полезнее для anti-fraud MVP.


## Что мы изучаем

На этом шаге мы изучаем не новые признаки, а разницу между простыми моделями.

Что важно понять:
- может ли нелинейная модель поймать сигнал, который не ловит `LogisticRegression`;
- улучшаются ли метрики без слишком сильного роста `manual_review_rate`;
- стоит ли двигаться дальше в сторону model comparison или сначала усиливать feature set.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)


In [ ]:
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
TRANSACTION_PATH = DATA_DIR / 'train_transaction.csv'
IDENTITY_PATH = DATA_DIR / 'train_identity.csv'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TRANSACTION_PATH exists:', TRANSACTION_PATH.exists())
print('IDENTITY_PATH exists:', IDENTITY_PATH.exists())


In [ ]:
def load_csv_if_exists(path: Path):
    if path.exists():
        print(f'Loaded: {path.name}')
        return pd.read_csv(path)
    print(f'File not found: {path}')
    return None


transactions = load_csv_if_exists(TRANSACTION_PATH)
identity = load_csv_if_exists(IDENTITY_PATH)
